# Appendix E - Reproducible machine learning

*This notebook contains all the sample code in Appendix E.*

## Outline

- [What reproducibility means](#What)
- [Control the sources of variation](#Sources)
- [Record the computational environment](#Environment)
- [Version code, data, and configuration](#Git)
- [Log experiments and results](#Log)
- [Persist complete models safely](#Persist)
- [A minimal reproducibility checklist](#Checklist)

Reproducibility is essential for verifying experimental results, comparing models fairly, and maintaining machine-learning systems over time. This appendix summarizes the main practices needed to reconstruct an experiment from its code, data, configuration, and recorded environment.

## What reproducibility means <a id="What"></a>

The terminology is not completely uniform across disciplines. In this book, we use the following practical distinction:
- **repeatability**: the same team reruns the same code, data, and environment and obtains the same result;
- **reproducibility**: another person can reconstruct the computational workflow and obtain the same result, up to expected numerical variability;
- **replicability**: an independent implementation or dataset supports the same scientific conclusion.

Exact equality is not always possible. Parallel execution, floating-point arithmetic, GPU kernels, library updates, and hardware differences may introduce small numerical variations. A reproducible experiment should therefore record not only its final score, but also all information needed to explain and reproduce the computational procedure.

## Control the sources of variation <a id="Sources"></a>

Many machine-learning operations are *randomized*, including dataset splitting, parameter initialization, mini-batch construction, sampling, and cross-validation. A fixed **seed** makes these operations repeatable.

In [1]:
import random
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


SEED = 42

random.seed(SEED)
rng = np.random.default_rng(SEED)


X, y = load_wine(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=SEED,
    n_jobs=1
)

Using the same seed is necessary but not always sufficient. Every component that performs random operations should be configured explicitly. For deep-learning frameworks, deterministic execution may also require framework-specific settings and can reduce computational performance.

> A fixed seed reproduces one realization of an experiment; it does not prove that the result is stable. When possible, report results over several independent seeds or repeated cross-validation runs.

The train, validation, and test partitions should be created once and stored or reconstructed from a fixed rule. Preprocessing and hyperparameter selection must use only the training data. In Scikit-learn, a `Pipeline` helps prevent leakage by fitting every transformation inside each training fold.

## Record the computational environment <a id="Environment"></a>

Different versions of Python and its libraries may produce different results or may be incompatible with saved models. Each project should therefore use an isolated environment and record its dependencies.

A **Conda environment** can be created and exported as follows:

For a \texttt{pip}-based project, the installed packages can be recorded with:

The Python version, operating system, hardware, and accelerator or driver versions should also be recorded when they can affect the result. A lightweight experiment header can be generated as follows:

In [2]:
import platform
import sys
import numpy as np
import sklearn

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
}

print(environment)

{'python': '3.9.18 (main, Sep 11 2023, 14:09:26) [MSC v.1916 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'numpy': '1.26.3', 'scikit_learn': '1.3.0'}


A *container*, such as a **Docker** image, provides stronger isolation by packaging the application, system libraries, and runtime configuration. Containers improve portability, but they do not by themselves version the data or guarantee deterministic hardware behavior.

## Version code, data, and configuration <a id="Git"></a>

Source code should be stored in a version-control system such as **Git**. Each meaningful experiment should be associated with a commit identifier, so that the exact implementation can be recovered later. Generated files, credentials, caches, and large temporary artifacts should be excluded through a suitable `.gitignore` file.

A minimal Git workflow is:

Large datasets and trained models should generally not be committed directly to **Git**. They can instead be stored in an artifact repository or managed by a **data-versioning tool** such as DVC. At minimum, record the data source, acquisition date, preprocessing procedure, split definition, and a **checksum** of each critical file:

In [3]:
from pathlib import Path
import hashlib

def sha256_file(filename, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(filename).open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()

print(sha256_file("../data/train.csv"))

f5d0c11e5c78a69a20dbb80baf2b24703f59a6687595752abb397d23732647c5


Hyperparameters and paths should be separated from the program logic. A simple **YAML** configuration may contain:

It should be loaded safely:

In [4]:
from pathlib import Path
import yaml

with Path("../data/config.yml").open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

seed = config["seed"]
n_estimators = config["model"]["n_estimators"]

The configuration file should be versioned together with the code.

## Log experiments and results <a id="Log"></a>

An experiment **log** should identify what was run, with which data and configuration, and what it produced. Useful fields include:
- date, duration, and experiment identifier;
- code commit and data version;
- random seed and hyperparameters;
- training, validation, and test metrics;
- warnings, failures, and output artifact paths.

Python's `logging` module provides a simple solution:

In [5]:
import logging

logging.basicConfig(
    filename="../data/experiment.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logging.info("Training started")
logging.info("seed=%d, n_estimators=%d", seed, n_estimators)

model.fit(X_train, y_train)
score = model.score(X_test, y_test)

logging.info("test_accuracy=%.4f", score)
logging.info("Training completed")

Metrics can also be stored in a machine-readable format:

In [6]:
import json
from pathlib import Path

results = {
    "seed": seed,
    "test_accuracy": float(score),
    "model": config["model"],
}

Path("../data/results.json").write_text(
    json.dumps(results, indent=2),
    encoding="utf-8"
)

132

For large projects, an experiment-tracking system can automate artifact storage, run comparison, and metadata collection. Regardless of the tool, the test-set result should be recorded only after model selection has been completed.

## Persist complete models safely <a id="Persist"></a>

When preprocessing is required, save the fitted pipeline rather than only the final estimator. This ensures that prediction uses the same transformations employed during training.

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path
import joblib


scaler = StandardScaler()

model = RandomForestClassifier(
    n_estimators=200,
    random_state=SEED,
    n_jobs=1
)

pipe = make_pipeline(scaler, model)
pipe.fit(X_train, y_train)

model_file = Path("../data/artifacts/model.joblib")
model_file.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(pipe, model_file)
loaded_pipe = joblib.load(model_file)

y_pred = loaded_pipe.predict(X_test)

> Both `pickle` and `joblib` may execute arbitrary code while loading an object. Never load a model from an untrusted source. A saved model may also be incompatible with a different Python or library version, so the environment metadata must be stored with the artifact.

The model file should be accompanied by its checksum, training configuration, class labels, expected input schema, software versions, evaluation results, and creation date. Sensitive information, passwords, and access tokens must never be stored in source repositories or experiment logs.

## A minimal reproducibility checklist <a id="Checklist"></a>

Before publishing, sharing, or deploying an experiment, verify that:
1. the code and configuration are versioned;
2. the data source, preprocessing, split, and checksum are recorded;
3. all random generators and randomized estimators are configured;
4. preprocessing is fitted only on training data;
5. the software environment is documented or containerized;
6. metrics, seeds, hyperparameters, and artifacts are logged;
7. the complete fitted pipeline is saved with its metadata;
8. the experiment can be rerun from a clean environment;
9. several seeds or resampling runs are used when variability matters.

Reproducibility is not a final action performed after training. It is a property of the entire workflow and should be designed into the project from the beginning.